## Silver Modeling Validation

## Step 1. Imports

In [0]:
from pyspark.sql import functions as F

from notebooks._shared.configuration import AppConfig
from notebooks._shared.contracts import SILVER_CONTRACTS

## Step 2. Configuração

In [0]:
dbutils.widgets.text("catalog", "movielakehouse")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")

config = AppConfig(
    catalog=dbutils.widgets.get("catalog"),
    bronze_schema=dbutils.widgets.get("bronze_schema"),
    silver_schema=dbutils.widgets.get("silver_schema"),
)

movies_bronze_table = f"{config.bronze_namespace}.movies"
credits_bronze_table = f"{config.bronze_namespace}.credits"

print(f"tabela bronze movies: {movies_bronze_table}")
print(f"tabela bronze credits: {credits_bronze_table}")
print(f"Namespace Silver: {config.silver_namespace}")

## Step 3. Inspeção dos contratos Silver

In [0]:
for contract in SILVER_CONTRACTS:
    print("=" * 50)
    print(f"Contrato: {contract.name}")
    print(f"Grão: {contract.grain}")
    print(f"Chave: {contract.key}")
    print("Schema:")

    for field in contract.schema.fields:
        print(
            f"  {field.name:<30} | "
            f"Tipo: {field.dataType.simpleString():<12} | "
            f"Nullable: {str(field.nullable):<5}"
        )
    print("=" * 50)
    print()

## Step 4. Inspeção dos campos semiestruturados

In [0]:
movies_bronze_df = spark.table(movies_bronze_table)
credits_bronze_df = spark.table(credits_bronze_table)

display(
    movies_bronze_df.select(
        "genres",
        "keywords",
        "production_companies",
        "production_countries",
        "spoken_languages",
    ).limit(2)
)

display(
    credits_bronze_df.select(
        "cast",
        "crew",
    ).limit(2)
)

## Step 5. Inferência estrutural dos campos semiestruturados

In [0]:
movies_json_fields = [
    "genres",
    "keywords",
    "production_companies",
    "production_countries",
    "spoken_languages",
]

credits_json_fields = [
    "cast",
    "crew",
]

print("Schemas observados em Movies:")

for field_name in movies_json_fields:
    inferred_schema = movies_bronze_df.select(
        F.expr(f"schema_of_json_agg({field_name})").alias("schema")
    ).first()["schema"]
    print(f"  {field_name}: {inferred_schema}")

print()

print("Schemas observados em Credits:")
for field_name in credits_json_fields:
    inferred_schema = credits_bronze_df.select(
        F.expr(f"schema_of_json_agg({field_name})").alias("schema")
    ).first()["schema"]
    print(f"  {field_name}: {inferred_schema}")

In [0]:
movies_json_fields = [
    "genres",
    "keywords",
    "production_companies",
    "production_countries",
    "spoken_languages",
]

credits_json_fields = [
    "cast",
    "crew",
]

# INFERÊNCIA DE CAMPOS SEMIESTRUTURADOS (MOVIES)

# monta todas as expressões do select dinamicamente
movies_exprs = [
    F.expr(f"schema_of_json_agg({field})").alias(field) for field in movies_json_fields
]

# roda um único select e captura a linha de resultado
movies_schemas_row = movies_bronze_df.select(*movies_exprs).first()

for field_name in movies_json_fields:
    print(f"  {field_name:<25}: {movies_schemas_row[field_name]}")

print()

#  INFERÊNCIA DE CAMPOS SEMIESTRUTURADOS (CREDITS)
print("Schemas observados em Credits:")

credits_exprs = [
    F.expr(f"schema_of_json_agg({field})").alias(field) for field in credits_json_fields
]

credits_schemas_row = credits_bronze_df.select(*credits_exprs).first()

for field_name in credits_json_fields:
    print(f"  {field_name:<25}: {credits_schemas_row[field_name]}")

In [0]:
# Movies e Credits atualmente disponíveis na Bronze pertencem ao mesmo _ingestion_id?
movies_ingestion_ids = movies_bronze_df.groupBy("_ingestion_id").count()
credits_ingestion_ids = credits_bronze_df.groupBy("_ingestion_id").count()

print(f"Movies Bronze:")
movies_ingestion_ids.show(truncate=False)

print("Credits Bronze:")
credits_ingestion_ids.show(truncate=False)

## Step 7. Validação do parsing dos campos semiestruturados

In [0]:
genres_schema = "ARRAY<STRUCT<id: BIGINT, name: STRING>>"

genres_parsed_df = movies_bronze_df.select(
    "id",
    "_ingestion_id",
    "genres",
    F.from_json("genres", genres_schema).alias("genres_parsed"),
)

display(
    genres_parsed_df.select("id","genres","genres_parsed").limit(2)
)

In [0]:
movies_nested_schemas = {
    "genres": "ARRAY<STRUCT<id: BIGINT, name: STRING>>",
    "keywords": "ARRAY<STRUCT<id: BIGINT, name: STRING>>",
    "production_companies": "ARRAY<STRUCT<id: BIGINT, name: STRING>>",
    "production_countries": "ARRAY<STRUCT<iso_3166_1: STRING, name: STRING>>",
    "spoken_languages": "ARRAY<STRUCT<iso_639_1: STRING, name: STRING>>",
}

credits_nested_schemas = {
    "cast": (
        "ARRAY<STRUCT<cast_id: BIGINT, character: STRING, credit_id: STRING, gender: BIGINT, id: BIGINT, name: STRING, order: BIGINT>>"
    ),
    "crew": (
        "ARRAY<STRUCT<credit_id: STRING, department: STRING, gender: BIGINT, id: BIGINT, job: STRING, name: STRING>>"
    ),
}

movies_parsed_df = movies_bronze_df.select(
    "*",
    *[
        F.from_json(F.col(column), schema).alias(f"{column}_parsed")
        for column, schema in movies_nested_schemas.items()
    ],
)

credits_parsed_df = credits_bronze_df.select(
    "*",
    *[
        F.from_json(F.col(column), schema).alias(f"{column}_parsed")
        for column, schema in credits_nested_schemas.items()
    ],
)

In [0]:
# Validar se o parsing produziu NULL onde a origem não era NULL

movies_parse_failures = movies_parsed_df.select(
    *[
        F.sum(
            F.when(
                F.col(column).isNotNull() & F.col(f"{column}_parsed").isNull(), 1
            ).otherwise(0)
        ).alias(column)
        for column in movies_nested_schemas
    ]
)

credits_parse_failures = credits_parsed_df.select(
    *[
        F.sum(
            F.when(
                F.col(column).isNotNull() & F.col(f"{column}_parsed").isNull(), 1
            ).otherwise(0)
        ).alias(column)
        for column in credits_nested_schemas
    ]
)

print("Movies com falhas de parsing:")
display(movies_parse_failures)

print("Credits com falhas de parsing:")
display(credits_parse_failures)

## Step 8. Validaão de modelagem das entidades Silver

### Step 8.1 Contrato: movie_genre

In [0]:
movies_genre_df = (
    movies_parsed_df
    .select(
        F.col("id").cast("bigint").alias("movie_id"),
        F.explode("genres_parsed").alias("genre"),
        "_ingestion_id"
    )
    .select(
        "movie_id",
        F.col("genre.id").alias("genre_id"),
        F.col("genre.name").alias("genre_name"),
        "_ingestion_id"
    )
)

display(movies_genre_df.limit(6))
movies_genre_df.printSchema()

In [0]:
# verifica a unicidade da chave definida movie_genre
movie_genre_duplicate_keys = (
    movies_genre_df.groupBy("movie_id", "genre_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Chaves duplicadas: {movie_genre_duplicate_keys}")

In [0]:
# verifica a obrigatoriedade dos campos definidos no contrato.
movie_genre_required_nulls = movies_genre_df.select(
    *[
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in ["movie_id", "genre_id", "genre_name", "_ingestion_id"]
    ]
)
display(movie_genre_required_nulls)

### Step 8.2 Contrato: movie

In [0]:
movie_df = movies_parsed_df.select(
    F.col("id").cast("bigint").alias("movie_id"),
    F.col("budget").cast("bigint").alias("budget"),
    F.col("homepage").cast("string").alias("homepage"),
    F.col("original_language").cast("string").alias("original_language"),
    F.col("original_title").cast("string").alias("original_title"),
    F.col("overview").cast("string").alias("overview"),
    F.col("popularity").cast("double").alias("popularity"),
    F.to_date("release_date").alias("release_date"),
    F.col("revenue").cast("bigint").alias("revenue"),
    F.col("runtime").cast("double").alias("runtime"),
    F.col("status").cast("string").alias("status"),
    F.col("tagline").cast("string").alias("tagline"),
    F.col("title").cast("string").alias("title"),
    F.col("vote_average").cast("double").alias("vote_average"),
    F.col("vote_count").cast("bigint").alias("vote_count"),
    "_ingestion_id",
)

movie_df.printSchema()
display(movie_df.limit(10))

In [0]:
# Verifica se existem valores nulos nas colunas requeridas
required_movie_columns = [
    "movie_id",
    "budget",
    "original_language",
    "original_title",
    "popularity",
    "revenue",
    "status",
    "title",
    "vote_average",
    "vote_count",
    "_ingestion_id",
]

movie_null_counts = movie_df.select(
    *[
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in required_movie_columns
    ]
)

display(movie_null_counts)

In [0]:
# verifa se conversões de tipo descartam valores não nulos da Bronze

movie_conversion_failures = movies_bronze_df.select(
    F.sum(
        F.when(
            F.col("id").isNotNull() & F.col("id").cast("bigint").isNull(),
            1,
        ).otherwise(0)
    ).alias("movie_id"),
    F.sum(
        F.when(
            F.col("budget").isNotNull() & F.col("budget").cast("bigint").isNull(),
            1,
        ).otherwise(0)
    ).alias("budget"),
    F.sum(
        F.when(
            F.col("popularity").isNotNull()
            & F.col("popularity").cast("double").isNull(),
            1,
        ).otherwise(0)
    ).alias("popularity"),
    F.sum(
        F.when(
            F.col("release_date").isNotNull() & F.to_date("release_date").isNull(),
            1,
        ).otherwise(0)
    ).alias("release_date"),
    F.sum(
        F.when(
            F.col("revenue").isNotNull() & F.col("revenue").cast("bigint").isNull(),
            1,
        ).otherwise(0)
    ).alias("revenue"),
    F.sum(
        F.when(
            F.col("runtime").isNotNull() & F.col("runtime").cast("double").isNull(),
            1,
        ).otherwise(0)
    ).alias("runtime"),
    F.sum(
        F.when(
            F.col("vote_average").isNotNull()
            & F.col("vote_average").cast("double").isNull(),
            1,
        ).otherwise(0)
    ).alias("vote_average"),
    F.sum(
        F.when(
            F.col("vote_count").isNotNull()
            & F.col("vote_count").cast("bigint").isNull(),
            1,
        ).otherwise(0)
    ).alias("vote_count"),
)

display(movie_conversion_failures)

In [0]:
# verifica a unicidade da chave definida no contrato
movie_duplicate_keys = (
    movie_df.groupBy("movie_id").count().filter(F.col("count") > 1).count()
)

print(f"Chaves duplicadas: {movie_duplicate_keys}")

### Step 8.3 Contrato: movie_keyword

In [0]:
# normaliza os keywords do filme conforme o contrato Silver.
movie_keyword_df = movies_parsed_df.select(
    F.col("id").cast("bigint").alias("movie_id"),
    F.explode("keywords_parsed").alias("keyword"),
    "_ingestion_id",
).select(
    "movie_id",
    F.col("keyword.id").alias("keyword_id"),
    F.col("keyword.name").alias("keyword_name"),
    "_ingestion_id",
)

display(movie_keyword_df.limit(6))
movie_keyword_df.printSchema()

In [0]:
# verifica a unicidade da chave definida no contrato movie_keyword
movie_keyword_duplicate_keys = (
    movie_keyword_df.groupBy("movie_id", "keyword_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Chaves duplicadas: {movie_keyword_duplicate_keys}")

In [0]:
# verifica a obrigatoriedade dos campos definidos no contrato
movie_keyword_required_nulls = movie_keyword_df.select(
    *[
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in ["movie_id", "keyword_id", "keyword_name", "_ingestion_id"]
    ]
)

display(movie_keyword_required_nulls)

### Step 8.4 Contrato: movie_production_company

In [0]:
# normaliza as produtoras do filme conforme o contrato Silver
movie_production_company_df = movies_parsed_df.select(
    F.col("id").cast("bigint").alias("movie_id"),
    F.explode("production_companies_parsed").alias("production_company"),
    "_ingestion_id",
).select(
    "movie_id",
    F.col("production_company.id").alias("company_id"),
    F.col("production_company.name").alias("company_name"),
    "_ingestion_id",
)

display(movie_production_company_df.limit(6))
movie_production_company_df.printSchema()

In [0]:
# verifica a unicidade da chave definida no contrato movie_production_company
movie_prod_company_duplicate_keys = (
    movie_production_company_df.groupBy("movie_id", "company_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Chaves duplicadas: {movie_prod_company_duplicate_keys}")

In [0]:
# verifica a obrigatoriedade dos campos definidos no contrato
movie_production_company_required_nulls = movie_production_company_df.select(
    *[
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in ["movie_id", "company_id", "company_name", "_ingestion_id"]
    ]
)

display(movie_production_company_required_nulls)

### Step 8.5 Contrato: movie_production_country

In [0]:
# normaliza os países de produção do filme conforme o contrato Silver
movie_production_country_df = movies_parsed_df.select(
    F.col("id").cast("bigint").alias("movie_id"),
    F.explode("production_countries_parsed").alias("production_country"),
    "_ingestion_id",
).select(
    "movie_id",
    F.col("production_country.iso_3166_1").alias("country_code"),
    F.col("production_country.name").alias("country_name"),
    "_ingestion_id",
)

display(movie_production_country_df.limit(5))
movie_production_country_df.printSchema()

In [0]:
# verifica a unicidade da chave definida no contrato movie_production_country
movie_prod_country_duplicate_keys = (
    movie_production_country_df.groupBy("movie_id", "country_code")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Chaves duplicadas: {movie_prod_country_duplicate_keys}")

In [0]:
# verifica a obrigatoriedade dos campos definidos no contrato
movie_production_country_required_nulls = movie_production_country_df.select(
    *[
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in ["movie_id", "country_code", "country_name", "_ingestion_id"]
    ]
)

display(movie_production_country_required_nulls)

### Step 8.6 Contrato: movie_spoken_language

In [0]:
# normaliza os idiomas falados do filme conforme o contrato Silver
movie_spoken_language_df = movies_parsed_df.select(
    F.col("id").cast("bigint").alias("movie_id"),
    F.explode("spoken_languages_parsed").alias("spoken_language"),
    "_ingestion_id",
).select(
    "movie_id",
    F.col("spoken_language.iso_639_1").alias("language_code"),
    F.col("spoken_language.name").alias("language_name"),
    "_ingestion_id",
)

display(movie_spoken_language_df.limit(5))
movie_spoken_language_df.printSchema()

In [0]:
# verifica a unicidade da chave definida no contrato movie_spoken_language
movie_spoke_lang_duplicate_keys = (
    movie_spoken_language_df.groupBy("movie_id", "language_code")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Chaves duplicadas: {movie_spoke_lang_duplicate_keys}")

In [0]:
# verifica a obrigatoriedade dos campos definidos no contrato
movie_spoken_language_required_nulls = movie_spoken_language_df.select(
    *[
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in ["movie_id", "language_code", "language_name", "_ingestion_id"]
    ]
)

display(movie_spoken_language_required_nulls)

### Step 8.7 Contrato: cast_credit

In [0]:
# normaliza os créditos de elenco conforme o contrato Silver
cast_credit_df = credits_parsed_df.select(
    F.col("movie_id").cast("bigint").alias("movie_id"),
    F.explode("cast_parsed").alias("cast_member"),
    "_ingestion_id",
).select(
    "movie_id",
    F.col("cast_member.credit_id").alias("credit_id"),
    F.col("cast_member.id").alias("person_id"),
    F.col("cast_member.name").alias("person_name"),
    F.col("cast_member.cast_id").alias("cast_id"),
    F.col("cast_member.character").alias("character"),
    F.col("cast_member.gender").alias("gender"),
    F.col("cast_member.order").alias("cast_order"),
    "_ingestion_id",
)

display(cast_credit_df.limit(5))
cast_credit_df.printSchema()

In [0]:
# verifica obrigatoriedade e unicidade da chave definida no contrato
cast_required_nulls = cast_credit_df.select(
    *[
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in [
            "movie_id",
            "credit_id",
            "person_id",
            "person_name",
            "_ingestion_id",
        ]
    ]
)

cast_duplicate_keys = (
    cast_credit_df.groupBy("movie_id", "credit_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(cast_required_nulls)
print(f"Chaves duplicadas: {cast_duplicate_keys}")

### Step 8.8 Contrato: crew_credit

In [0]:
# normaliza os créditos da equipe técnica conforme o contrato Silver
crew_credit_df = credits_parsed_df.select(
    F.col("movie_id").cast("bigint").alias("movie_id"),
    F.explode("crew_parsed").alias("crew_member"),
    "_ingestion_id",
).select(
    "movie_id",
    F.col("crew_member.credit_id").alias("credit_id"),
    F.col("crew_member.id").alias("person_id"),
    F.col("crew_member.name").alias("person_name"),
    F.col("crew_member.gender").alias("gender"),
    F.col("crew_member.department").alias("department"),
    F.col("crew_member.job").alias("job"),
    "_ingestion_id",
)

display(crew_credit_df.limit(5))
crew_credit_df.printSchema()

In [0]:
# Verifica a obrigatoriedade dos campos definidos no contrato.
crew_required_nulls = crew_credit_df.select(
    *[
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in [
            "movie_id",
            "credit_id",
            "person_id",
            "person_name",
            "department",
            "job",
            "_ingestion_id",
        ]
    ]
)

display(crew_required_nulls)

In [0]:
# verifica a unicidade da chave definida no contrato crew_credit
crew_duplicate_keys = (
    crew_credit_df.groupBy("movie_id", "credit_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Chaves duplicadas: {crew_duplicate_keys}")

## Step 9. Estratégia de materialização e recuperação da Silver

Valida se as entidades Silver podem ser reprocessadas integralmente após uma falha parcial, sem duplicação ou alteração indevida do resultado esperado

In [0]:
# registra a quantidade de linhas produzida por cada entidade Silver.
silver_dataframes = {
    "movie": movie_df,
    "movie_genre": movies_genre_df,
    "movie_keyword": movie_keyword_df,
    "movie_production_company": movie_production_company_df,
    "movie_production_country": movie_production_country_df,
    "movie_spoken_language": movie_spoken_language_df,
    "cast_credit": cast_credit_df,
    "crew_credit": crew_credit_df,
}

silver_reference_counts = {
    entity: dataframe.count() for entity, dataframe in silver_dataframes.items()
}

for entity, count in silver_reference_counts.items():
    print(f" {entity:<30}: {count}")

### Step 9.2. Simulação de falha parcial

Simula uma interrupção durante a materialização das entidades para observar o estado deixado por uma execução parcialmente concluída

In [0]:
# define tabelas isoladas para o experimento de recuperação.
recovery_tables = {
    "movie": f"{config.silver_namespace}._recovery_test_movie",
    "movie_genre": f"{config.silver_namespace}._recovery_test_movie_genre",
    "movie_keyword": f"{config.silver_namespace}._recovery_test_movie_keyword",
}

recovery_tables

In [0]:
# simula uma execução interrompida após duas materializações concluídas
movie_df.write.mode("overwrite").saveAsTable(recovery_tables["movie"])

movies_genre_df.write.mode("overwrite").saveAsTable(recovery_tables["movie_genre"])

raise RuntimeError("Falha parcial simulada antes de movie_keyword.")